# Valeria — NeuroMatch 2026 · Posterior Motives

**Your slide's questions:**

- Compare **prior concentration / prior probability** from Switch vs hierarchical.
- How does it change across coherence and prior SD?
- Switch: **one** prior weight per block. Hierarchical: varies **trial by trial**.
- Final plot comparing what both models output as 'reliance on the prior'.
- ANOVA (coherence + prior_std). (This section comes after Romi's.)

This notebook is runnable top-to-bottom and uses **HB-Rachel** (our hierarchical Bayesian observer) and the **Switch** (paper's switching observer). Edit `SUBJECT` / filters and re-run any cell. See the API guide cell for how to make your own calls.

In [ ]:
# ============================================================
#  SETUP  —  works in Google Colab or on a local checkout
# ============================================================
import os, sys, subprocess

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
BRANCH   = "model-verification"     # branch that carries the fitted models + API

def _find_root():
    """If we're already inside a checkout, use it (no clone needed)."""
    here = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "observers", "api.py")):
            return here
        here = os.path.dirname(here)
    return None

ROOT = _find_root()
if ROOT is None:
    # Colab path: shallow-clone the repo, then point at the hierarchical/ package.
    dest = "/content/NeuroMatch_2026_Behaviour" if os.path.isdir("/content") \
           else os.path.abspath("NeuroMatch_2026_Behaviour")
    if not os.path.isdir(dest):
        # PUBLIC repo: this just works. PRIVATE repo: replace REPO_URL with
        #   https://<TOKEN>@github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                        REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, "hierarchical")

sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from observers import api

print("repo root :", ROOT)
print("models    :", api.list_models())
print("data      :", "data/data01_direction4priors.csv  (12 subjects)")

# figures for this notebook go in a dedicated subfolder beside it
FIG_DIR = os.path.join(ROOT, "experiments", "valeria", "01_slide_notebook", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("figures  :", FIG_DIR)

## How to use the model API (read me — also for an LLM assistant)

Everything below comes from the **fitted models**, reached through one module:
`observers.api`. You never touch raw model math — you call functions.

### Where the data lives (relative to the repo's `hierarchical/` folder)
| what | path |
|---|---|
| trial-level data (all 12 subjects) | `data/data01_direction4priors.csv` |
| point fits (per model, per subject) | `results/fits/comparison/<model>/subject<N>.json` |
| cross-validation records | `results/fits/comparison_cv/<model>/subject<N>_cv.json` |
| project abstract | `docs/project_abstract.md` |

### Model keys
`'switch'` (paper's Switching observer, 9 params, non-learning) · `'hb_rachel'`
(our hierarchical Bayesian observer, 7 params, **learns** prior width online) ·
also available: `'hb_adaptive'`, `'hb_salma'`, `'recombined'`, `'basic_bayes'`.

### The API — five kinds of call
```python
from observers import api

# LOAD --------------------------------------------------------------
api.load_subject(2)                 # -> DataFrame: motion_direction, motion_coherence,
                                    #    prior_std, estimate_dir  (one subject's trials)
api.load_fitted('hb_rachel', 2)     # -> (observer, record) with fitted params
api.observed_distribution(2, direction=335, coherence=0.06, prior_std=80)
                                    # -> empirical response histogram (density over 1..360)

# INSPECT -----------------------------------------------------------
api.list_models(); api.model_info() # what exists, params, colors
api.fitted_subjects('hb_rachel')    # which subjects are fit

# PREDICT -----------------------------------------------------------
api.predict('hb_rachel', 2)         # -> (n_trials, 360) predicted distribution per trial
api.belief_trajectory('hb_rachel', 2)
                                    # -> DataFrame trial, believed_sd  (the LEARNED prior width)

# FIT / SIMULATE ----------------------------------------------------
api.fit_model('hb_rachel', 2)       # refit from scratch (slow)
api.simulate('hb_rachel', 2, seed=0)# generative: synthetic responses from the fitted model

# EVALUATE ----------------------------------------------------------
api.results_table()                 # -> tidy DataFrame: model,label,subject,k,nll,aic,bic,cv_nll
api.trial_logliks('hb_rachel', 2)   # -> per-trial log-likelihood (slice it however you like)
api.bias_variability(2)             # -> per-condition estimation bias + circular SD (Fig-3 core)
```

### Custom calls (when a helper doesn't exist)
The API returns raw numbers; **you** aggregate/plot. To reach the model object
directly:
```python
from observers.comparison.registry import build_registry, load_subject
spec = build_registry(['hb_rachel'])['hb_rachel']   # a ModelSpec
obs, rec = api.load_fitted('hb_rachel', 2)
dists = spec.predict_distributions(obs, load_subject(2))  # (n_trials, 360)
```
Condition labels for any trial-level array come from `api.load_subject(s)` —
they're **row-aligned** with `predict`, `trial_logliks`, and `belief_trajectory`.

## 1 · κ means different things in the two models

κ is the **concentration** (inverse width) of a von Mises: high κ = narrow / confident, low κ = broad. Both models carry prior-related κ, but they are structurally different:

- **Switch** — the prior concentration `k_prior` is a **fixed lookup per block** (one value for each prior SD), set once by fitting.
- **HB-Rachel** — the believed prior concentration is **learned and updated every trial**; there is no fixed per-block value, only a trajectory.

So the comparison is: Switch's *one number per block* vs HB-Rachel's *trial-by-trial belief*, both expressed as a prior width.

In [ ]:
# ---- shared helper: learned belief joined with per-trial conditions ----
# belief_trajectory() replays the FITTED observer over the subject's real trials
# and returns the learned prior width (believed_sd) per trial. We join it with
# the condition labels (coherence, prior_std), which are ROW-ALIGNED with it.
def belief_with_conditions(key, subject):
    bt = api.belief_trajectory(key, subject)          # trial, believed_sd
    d  = api.load_subject(subject).reset_index(drop=True)
    out = bt.copy()
    out['coherence'] = d['motion_coherence'].values
    out['prior_std'] = d['prior_std'].values
    out['subject']   = subject
    return out

def all_beliefs(key, subjects=range(1,13)):
    return pd.concat([belief_with_conditions(key, s) for s in subjects],
                     ignore_index=True)

# HB-Rachel: learned prior width (from believed kappa) per trial, by block
allb = all_beliefs('hb_rachel')
hb_by_block = allb.groupby('prior_std').believed_sd.mean().round(2)
print('HB-Rachel mean LEARNED prior SD by block (trial-by-trial, averaged):')
print(hb_by_block.to_string())

## 2 · Switch's fixed per-block prior concentration

Switch stores `k_prior` as one value per prior-SD block. We convert each to the equivalent prior SD (via the same von Mises SD formula the models use) so it's on the same axis as HB-Rachel's learned width.

In [ ]:
import json
from observers.helpers.circular import von_mises_std
# Switch's fitted k_prior per block. Use the MEDIAN across subjects, not the mean:
# a few subjects fit huge k_prior (near-delta priors, e.g. k>900), which drags the
# mean and makes the implied SD collapse. Median is the robust per-block summary.
kp = {b: [] for b in ['10','20','40','80']}
for s in range(1,13):
    p = json.load(open(f'results/fits/comparison/switch/subject{s}.json'))['params']['k_prior']
    for b in kp: kp[b].append(p[b])
sw_sd = {int(b): von_mises_std(np.median(v), 225.0) for b,v in kp.items()}
sw_by_block = pd.Series(sw_sd).sort_index().round(2)
print('Switch fixed per-block prior SD (from MEDIAN k_prior across subjects):')
print(sw_by_block.to_string())
print('(median, not mean: a few subjects fit near-delta priors k>900 that skew the mean)')

## 3 · How prior reliance changes with coherence & prior SD

For HB-Rachel the believed width can in principle vary with coherence; for Switch `k_prior` is fixed per block regardless of coherence. This table lays both out.

In [ ]:
# HB-Rachel learned SD by coherence x prior_std
hb_tab = allb.groupby(['coherence','prior_std']).believed_sd.mean().round(1).unstack('prior_std')
print('HB-Rachel learned prior SD (coherence x block):'); print(hb_tab.to_string())
print('\nSwitch prior SD is coherence-INDEPENDENT (one value per block):')
print(sw_by_block.to_string())

## 4 · Final plot — Switch (fixed) vs HB-Rachel (learned)

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.6))
blocks = [10,20,40,80]
ax.plot(blocks, [sw_by_block[b] for b in blocks], 's--', color='#30638e', lw=2,
        label='Switch: fixed k_prior per block')
ax.plot(blocks, [hb_by_block[b] for b in blocks], 'o-', color='#edae49', lw=2,
        label='HB-Rachel: learned (trial-by-trial)')
ax.plot(blocks, blocks, 'k:', alpha=0.5, label='true block SD')
ax.set_xlabel('nominal prior SD (deg)'); ax.set_ylabel('prior SD the model uses (deg)')
ax.set_title('Reliance on the prior: fixed lookup (Switch) vs learned belief (HB-Rachel)')
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'valeria_kappa_comparison.png'), dpi=130); plt.show()

## 5 · ANOVA on the learned prior width (coherence + prior_std)

In [ ]:
# ---- shared helper: two-way ANOVA believed_sd ~ coherence + prior_std ----
# Uses PER-SUBJECT cell means (not raw trials) to avoid pseudoreplication.
import statsmodels.api as sm
import statsmodels.formula.api as smf
def anova_believed_sd(df):
    cell = df.groupby(['subject','coherence','prior_std']).believed_sd.mean().reset_index()
    model = smf.ols('believed_sd ~ C(coherence) + C(prior_std)', data=cell).fit()
    aov = sm.stats.anova_lm(model, typ=2)
    aov['eta_sq'] = aov['sum_sq'] / aov['sum_sq'].sum()
    return aov[['sum_sq','F','PR(>F)','eta_sq']].round(4)

print(anova_believed_sd(allb).to_string())
print('\nInterpretation: for HB-Rachel the learned prior width is driven almost')
print('entirely by prior_std (block width), NOT coherence — the same block')
print('dependence Switch bakes in as a fixed lookup, but here it is LEARNED.')
print('That is the reframing: a discrete per-block weight becomes a graded,')
print('feedback-driven belief that lands in the same place.')